In [7]:
# Ignore the warnings
import warnings
# warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

# System related and data input controls
import os

# Python path
import sys
base_folder = 'DataScience'
location_base = os.path.join(os.getcwd().split(base_folder)[0], base_folder)
location_module = [os.path.join(location_base, 'Module')] 
for each in location_module:
    if each not in sys.path:
        sys.path.append(each)

# Auto reload of library
%reload_ext autoreload
%autoreload 2

from import_KK import *
# DeviceStrategy_CPU()
DeviceStrategy_GPU()
from preprocessing_KK import *
from preprocessing_project_KK import *
from description_KK import *
from algorithm_machinelearning_KK import *
from algorithm_deeplearning_KK import *
from evaluation_KK import *
from visualization_KK import *


=========== GPU Strategy ===========
Detected GPUs:
 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
TF using MirroredStrategy with 2 GPUs:
 ['/GPU:0', '/GPU:1']


======= GPU / CUDA / STATUS ========
Cuda Ready?        True
CUDA Version:      12.5.1
cuDNN Version:     9

TF Version:        2.19.0
Keras Version:     3.10.0
True
Torch Version:       2.7.1+cu126
Torch CUDA Version:  12.6
Torch cuDNN Version: 90501

Torch GPUs Available: 2
Use the GPU: NVIDIA GeForce RTX 3090



# Hyperparameters

In [26]:
# Data
FOLDER_GDM, FOLDER_GDS = get_google_drivelocation(gdrive='z', glocation=os.path.join('Research', 'SavedData', 'REES46'))
FOLDER_LOCATION = FOLDER_GDM
FILENAME = 'featured_causal_balanced.parquet'
FILE_LOCATION = os.path.join(FOLDER_LOCATION, FILENAME)
LANGUAGE = 'kr'

# Preprocessing
Y_colname = 'is_purchased'
T_colname = 'ab_test'
X_colname_dict = {
    'baseline': [
        'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
        'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
        'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
        'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
        'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
        'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
        'channel_email', 'channel_mobile_push', 'channel_web_push',
        'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
        'is_holiday', 'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
        'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
        'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
        'total_campaigns', 'total_messages', 'total_purchases'
    ],
    # 최근성 및 위험도 (Recency & Risk)
    'fe_recency_risk': [
        'days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac'
    ],
    # 캘린더 및 시점 이펙트 (Calendar & Timing Effects)
    'fe_calendar_timing': [
        'cal_is_weekend', 'cal_week_of_month', 'feat_dow_shift',
        'feat_eoq_bump', 'feat_hour_shift', 'feat_payday_bump'
    ],
    # 피로도 및 유저 반응성 (Fatigue & User Responsiveness)
    'fe_fatigue_response': [
        'ctx_tc_open_rate_30d', 'feat_fatigue', 'feat_last_any_hours',
        'feat_last_email_hours', 'feat_last_mobile_push_hours',
        'u_cadence_std_30d', 'u_click_rate_30d', 'u_open_cnt_30d', 'u_open_rate_30d'
    ],
    # 토픽 및 선호도 일치성 (Topic & Preference Alignment)
    'fe_topic_preference': [
        'feat_like_last_success', 'feat_path_align', 'feat_topic_novelty',
        'topic_N7', 'topic_t_since_hours'
    ]
}
X_nullmax = None
X_nullmin = None
X_dummy = None
X_encoding = None
X_delYrelated = None
X_del = None

TEST_SIZE = 0.2
RANDOM_STATE = 123
CLASS_STAT = True
SCALER = 'minmax'
LABEL_LIST = ['Non-purchased', 'Purchased']

# Model
## Causal
if T_colname != None:
    CAUSAL_CRITERION = 'het'
    CAUSAL_HONEST = True
    CAUSAL_INFERENCE = True

# Save
SAVE_NAME_PREDSTATTR = 'DescriptiveStatistics_BinaryPredTrain.csv'
SAVE_NAME_PREDSTATTE = 'DescriptiveStatistics_BinaryPredTest.csv'
SAVE_NAME_ML = 'Performance_ML'
SAVE_NAME_DL = 'Performance_DL'
SAVE_NAME_PERFORMANCE = 'Performance.csv'
SAVE_FOLDER_BASE = os.path.join('.', 'Result', 'Base')
SAVE_FOLDER_TEST = os.path.join('.', 'Result', 'Test')

# Data

In [18]:
def preprocessing_CheckDataLeakage(df, feature_cols, target_col, threshold=0.5):
    """
    Data Leakage(데이터 누수)가 의심되는 변수를 찾아내는 함수
    """
    print("\n[*] Checking for Data Leakage (미래 데이터 혼입 검증)...")
    suspects = []

    # 1. Target과의 단순 상관계수 확인 (Correlation)
    corr_matrix = df[feature_cols + [target_col]].corr()
    target_corr = corr_matrix[target_col].abs().drop(target_col)
    
    high_corr_features = target_corr[target_corr > threshold]
    if not high_corr_features.empty:
        print("    🚨 [의심 1] Target과 상관관계가 비정상적으로 높은 변수:")
        for feat, val in high_corr_features.items():
            print(f"       - {feat}: 상관계수 {val:.3f}")
            suspects.append(feat)
    else:
        print("    ✅ Target과의 비정상적인 선형 상관관계를 가진 변수 없음.")

    # 2. RandomForest를 이용한 Feature Importance 확인 (비선형 관계 포착)
    # [수정 포인트] 이미 메인 함수에서 Inf 처리를 하고 넘어왔으므로 안전합니다.
    df_sample = df.sample(n=min(10000, len(df)), random_state=42).fillna(0)
    X_sample = df_sample[feature_cols]
    Y_sample = df_sample[target_col].astype(int)

    rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf.fit(X_sample, Y_sample)

    importances = pd.Series(rf.feature_importances_, index=feature_cols)
    leaked_features = importances[importances > 0.40] 

    if not leaked_features.empty:
        print("    🚨 [의심 2] Target 예측 기여도가 비정상적으로 높은 변수:")
        for feat, val in leaked_features.items():
            print(f"       - {feat}: 중요도 {val:.1%}")
            if feat not in suspects:
                suspects.append(feat)
    else:
        print("    ✅ 예측 기여도를 독식하는 비정상적인 변수 없음.")
        
    return list(set(suspects))

def preprocessing_CausalForest(df_raw, X_colname_dict, 
                               Y_colname='is_purchased', T_colname='ab_test', 
                               time_colname=None, year_val=1, year_test=1, 
                               test_size=0.2, 
                               scaler='minmax', random_state=123,
                               auto_drop_leakage=True):
    start_time = time.time()
    df = df_raw.copy()
    print("="*65)
    print(" 🛠️ CAUSAL INFERENCE DATA PREPARATION")
    print("="*65)
    print(f"[*] Initial Shape: {df.shape}")

    # Feature 입력 유연하게 파싱 (List or Dict)
    if isinstance(X_colname_dict, dict):
        X_colname = list(set(sum(X_colname_dict.values(), [])))
        print(f"[*] Dictionary 입력 확인: 총 {len(X_colname)}개의 고유 변수 추출.")
    elif isinstance(X_colname_dict, list):
        X_colname = list(set(X_colname_dict))
        print(f"[*] List 입력 확인: 총 {len(X_colname)}개의 고유 변수 추출.")
    else:
        raise ValueError("[!] X_colname_dict은 List 또는 Dictionary 형태여야 합니다.")

    X_colname.sort()
    original_feature_count = len(X_colname)
    X_colname = [c for c in X_colname if c in df.columns]
    
    if len(X_colname) < original_feature_count:
        print(f"[*] Warning: Dataset에 없는 {original_feature_count - len(X_colname)}개의 변수를 제외했습니다.")

    # Data Leakage 검증 및 Treatment 처리
    df = df.replace([np.inf, -np.inf], np.nan)
    suspect_features = preprocessing_CheckDataLeakage(df, X_colname, Y_colname, threshold=0.5)
    
    if suspect_features and auto_drop_leakage:
        print(f"[*] Action: Leakage 의심 변수 {len(suspect_features)}개를 자동 제거합니다.")
        X_colname = [f for f in X_colname if f not in suspect_features]
            
    print("\n[*] Treatment(T) 결측치: 비즈니스 룰에 따라 0(Control)으로 대체합니다.")
    df[T_colname] = df[T_colname].fillna(0).astype(int)

    # X 결측치 처리 (5% Rule)
    initial_rows = len(df)
    if df[Y_colname].isna().sum() > 0:
        df = df.dropna(subset=[Y_colname]).reset_index(drop=True)
        
    rows_with_nan = df[X_colname].isna().any(axis=1).sum()
    nan_ratio = rows_with_nan / len(df)
    print(f"[*] Feature 결측치 비율: {nan_ratio:.1%}")
    
    if 0 < nan_ratio < 0.05:
        df = df.dropna(subset=X_colname).reset_index(drop=True)
    elif nan_ratio >= 0.05:
        df[X_colname] = df[X_colname].fillna(-999)
        
    if initial_rows - len(df) > 0:
        print(f"[*] 결측치 처리로 인해 {initial_rows - len(df):,}행이 제거되었습니다. (Shape: {df.shape})")

    # Train / Test 데이터 분할 (Time Split vs Random Split)
    print("\n[*] Data Splitting...")
    if test_size == 'Time':
        if time_colname is None or time_colname not in df.columns:
            raise ValueError("[!] test_size='Time'을 사용하려면 time_colname을 명시해야 합니다.")
            
        unique_times = sorted(df[time_colname].dropna().unique())
        test_threshold = unique_times[-year_test]
        val_threshold = unique_times[-(year_test + year_val)]
        
        print(f"    [OOT Split] Dynamic Cutoff using '{time_colname}'")
        print(f"    - Train : ~ {val_threshold - 1}")
        print(f"    - Val   : {val_threshold} ~ {test_threshold - 1}")
        print(f"    - Test  : {test_threshold} ~")
        
        train_val_mask = df[time_colname] < test_threshold
        test_mask = df[time_colname] >= test_threshold
        
        X_train = df[train_val_mask][X_colname]
        T_train = df[train_val_mask][T_colname]
        Y_train = df[train_val_mask][Y_colname]
        
        X_test = df[test_mask][X_colname]
        T_test = df[test_mask][T_colname]
        Y_test = df[test_mask][Y_colname]
        
        # OOT 교차 검증 인덱스 생성
        train_idx = np.where(df[train_val_mask][time_colname] < val_threshold)[0].tolist()
        val_idx = np.where(df[train_val_mask][time_colname] >= val_threshold)[0].tolist()
        cv_splits = (train_idx, val_idx)
        print(f"    ▷ [CV Tuple] Train: {len(train_idx)}, Val: {len(val_idx)}")
        
    else:
        # 무작위 분할 시에는 처치군(T) 비율을 유지(Stratify)
        X = df[X_colname]
        T = df[T_colname]
        Y = df[Y_colname]
        X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
            X, T, Y, test_size=test_size, random_state=random_state, stratify=T
        )
        cv_splits = None

    print(f"    - Train Shape: X{X_train.shape}, Y{Y_train.shape}")
    print(f"    - Test  Shape: X{X_test.shape}, Y{Y_test.shape}")

    # 스케일링
    if scaler is not None:
        print("\n[*] Preprocessing of Scaling...")
        scaler_dict = {}
        
        X_train_np = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
        X_test_np = X_test.values if isinstance(X_test, pd.DataFrame) else X_test
        
        X_train_scaled = np.zeros_like(X_train_np, dtype=float)
        X_test_scaled = np.zeros_like(X_test_np, dtype=float)
        
        for i, col in enumerate(X_colname):
            s = MinMaxScaler() if scaler == 'minmax' else StandardScaler()
            
            train_col = X_train_np[:, i].reshape(-1, 1)
            test_col = X_test_np[:, i].reshape(-1, 1)
            
            X_train_scaled[:, i] = s.fit_transform(train_col).flatten()
            X_test_scaled[:, i] = s.transform(test_col).flatten()
            scaler_dict[col] = s
            
        X_train = X_train_scaled
        X_test = X_test_scaled

    sec = time.time() - start_time
    print(f"\n[*] Complete! (소요 시간: {str(datetime.timedelta(seconds=sec)).split('.')[0]})")
    print("="*65)
    
    return {
        'X_train': X_train, 'X_test': X_test,
        'T_train': T_train.values, 'T_test': T_test.values,
        'Y_train': Y_train.values, 'Y_test': Y_test.values,
        'X_colname': X_colname,
        'scaler_dict': scaler_dict if scaler else None,
        'cv_splits': cv_splits,
        'df_raw': df # 필요시 원본 접근용
    }

In [27]:
df = pd.read_parquet(FILE_LOCATION)
df_prep = preprocessing_CausalForest(df, X_colname_dict, Y_colname, T_colname, 
                                     TEST_SIZE, SCALER, RANDOM_STATE)

 🛠️ CAUSAL INFERENCE DATA PREPARATION
[*] Initial Shape: (527618, 139)
[*] Dictionary 입력 확인: 총 64개의 고유 변수 추출.

[*] Checking for Data Leakage (미래 데이터 혼입 검증)...
    🚨 [의심 1] Target과 상관관계가 비정상적으로 높은 변수:
       - camp_campaign_typebulk: 상관계수 0.591
       - camp_topicother: 상관계수 0.570
       - camp_topicsale.out: 상관계수 0.570
       - message_type_bulk: 상관계수 0.591
       - platform.: 상관계수 0.723
    ✅ 예측 기여도를 독식하는 비정상적인 변수 없음.
[*] Action: Leakage 의심 변수 5개를 자동 제거합니다.

[*] Treatment(T) 결측치: 비즈니스 룰에 따라 0(Control)으로 대체합니다.
[*] Feature 결측치 비율: 1.9%
[*] 결측치 처리로 인해 9,820행이 제거되었습니다. (Shape: (517798, 139))

[*] Data Splitting...
    - Train Shape: X(414238, 59), Y(414238,)
    - Test  Shape: X(103560, 59), Y(103560,)

[*] Preprocessing of Scaling...

[*] Complete! (소요 시간: 0:00:05)


# Modeling

In [14]:
# sample_size = min(50000, len(df_prep['X_train']))
# sample_idx = np.random.choice(len(df_prep['X_train']), sample_size, replace=False)

# X_train_fast = df_prep['X_train'][sample_idx]
# T_train_fast = df_prep['T_train'][sample_idx]
# Y_train_fast = df_prep['Y_train'][sample_idx]

# print(f"[*] 원본 {len(df_prep['X_train']):,}개 중 {sample_size:,}개 샘플링 완료.")

# # Training
# model = CausalForest(
#     n_estimators=1000, 
#     criterion=CAUSAL_CRITERION, 
#     honest=CAUSAL_HONEST, 
#     inference=CAUSAL_INFERENCE, 
#     random_state=RANDOM_STATE,
#     n_jobs=-1
# )
# model.fit(X_train_fast, T_train_fast, Y_train_fast)

In [15]:
# Training
model = CausalForest(
    n_estimators=5000, 
    criterion=CAUSAL_CRITERION, 
    honest=CAUSAL_HONEST, 
    inference=CAUSAL_INFERENCE, 
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(df_prep['X_train'], df_prep['T_train'], df_prep['Y_train'])

# Predict
try:
    pred_results = model.predict(df_prep['X_test'], interval=CAUSAL_INFERENCE, alpha=0.05)
    if isinstance(pred_results, tuple) and len(pred_results) == 3:
        cate_pred, lb, ub = pred_results
        cate_pred, lb, ub = cate_pred.flatten(), lb.flatten(), ub.flatten()
    else:
        cate_pred = pred_results.flatten()
        lb, ub = cate_pred - 0.05, cate_pred + 0.05 # Fallback
except Exception as e:
    cate_pred = model.predict(df_prep['X_test']).flatten()
    lb, ub = cate_pred - 0.05, cate_pred + 0.05
print(f"[*] 평균 인과효과(ATE): {np.mean(cate_pred):.4f}")

,n_estimators,5000
,criterion,'het'
,max_depth,None
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,min_var_fraction_leaf,None
,min_var_leaf_on_val,False
,max_features,'auto'
,min_impurity_decrease,0.0
,max_samples,0.45


# Result

In [17]:
def plot_CausalForest(model, X_train, T_train, X_test, T_test, Y_test, 
                      feature_names, cate_pred, lb, ub,  
                      random_state=123, output_dir="Result"):
    """
    Causal Forest 학습 결과를 분석하기 위한 올인원 시각화 함수
    """
    os.makedirs(output_dir, exist_ok=True)
    sns.set_theme(style="whitegrid")
    print(f"\n{'='*50}\n[*] GENERATING CAUSAL VISUALIZATIONS\n{'='*50}")

    # ---------------------------------------------------------
    # [Fig 1] Positivity (Overlap) Assumption
    # 통제군(T=0)과 실험군(T=1)의 성향 점수 분포 비교
    # ---------------------------------------------------------
    try:
        print("[1/5] Overlap Assumption 체크 (Propensity Score)...")
        prop_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=random_state)
        prop_model.fit(X_train, T_train)
        propensity_scores = prop_model.predict_proba(X_test)[:, 1]

        plt.figure(figsize=(8, 5))
        sns.histplot(x=propensity_scores, hue=T_test, bins=50, kde=True, palette="Set1", alpha=0.6, log_scale=(False, True))
        plt.title("Propensity Score Overlap (Log Scale for Y-axis)", fontsize=14, fontweight='bold')
        plt.xlabel("Probability of Receiving Treatment P(T=1|X)")
        plt.ylabel("Count (Log Scale)")
        plt.savefig(f"{output_dir}/1_overlap_plot.png", dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  -> [!] Overlap 플롯 생성 실패: {e}")

    # ---------------------------------------------------------
    # [Fig 2] CATE Distribution & Confidence Intervals
    # 유저별 인과효과 분포 (효과가 높은 순으로 정렬)
    # ---------------------------------------------------------
    try:
        print("[2/5] 이질적 인과효과(CATE) 및 신뢰구간 분포 시각화...")
        # 데이터가 너무 크면 그리기 힘드므로 1,000개만 샘플링하여 시각화
        sample_size = min(1000, len(cate_pred))
        sample_indices = np.random.choice(len(cate_pred), sample_size, replace=False)
        
        c_sample, l_sample, u_sample = cate_pred[sample_indices], lb[sample_indices], ub[sample_indices]
        sort_idx = np.argsort(c_sample)

        plt.figure(figsize=(10, 6))
        plt.errorbar(np.arange(sample_size), c_sample[sort_idx],
                     yerr=[c_sample[sort_idx] - l_sample[sort_idx], u_sample[sort_idx] - c_sample[sort_idx]],
                     fmt='o', markersize=3, alpha=0.6, color='teal', ecolor='lightgray', capsize=0)
        plt.axhline(0, color='crimson', linestyle='--', lw=2)
        plt.xlabel(f"Sampled Users (Sorted by CATE, n={sample_size})", fontsize=12)
        plt.ylabel("Estimated CATE with 95% CI", fontsize=12)
        plt.title("Heterogeneous Treatment Effects (CATE)", fontsize=14, fontweight='bold')
        plt.savefig(f"{output_dir}/2_cate_distribution.png", dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  -> [!] CATE 분포 플롯 생성 실패: {e}")

    # ---------------------------------------------------------
    # [Fig 3] Feature Importance
    # 어떤 변수가 인과효과(CATE) 차이를 만드는데 가장 중요했는가?
    # ---------------------------------------------------------
    try:
        print("[3/5] Feature Importance 추출 중...")
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_() if callable(model.feature_importances_) else model.feature_importances_
            
            # 상위 15개 변수만 시각화
            sorted_idx = np.argsort(importances)[::-1][:15]
            top_imps = importances[sorted_idx]
            top_feats = [feature_names[i] for i in sorted_idx]
            
            plt.figure(figsize=(8, 6))
            sns.barplot(x=top_imps, y=top_feats, hue=top_feats, palette="viridis", legend=False)
            plt.title("Top 15 Feature Importance for Heterogeneity", fontsize=14, fontweight='bold')
            plt.xlabel("Importance Score")
            plt.savefig(f"{output_dir}/3_feature_importance.png", dpi=300, bbox_inches='tight')
            plt.close()
    except Exception as e:
        print(f"  -> [!] Feature Importance 플롯 생성 실패: {e}")

    # ---------------------------------------------------------
    # [Fig 4] SHAP Values (Surrogate Model 방식 적용)
    # ---------------------------------------------------------
    try:
        import shap
        from sklearn.ensemble import RandomForestRegressor
        print("[4/5] Surrogate Model을 활용한 SHAP Value 시각화...")
        
        # 1. Causal Forest가 예측한 CATE를 타겟으로 하는 설명용 '대리 모델' 학습
        # 해석이 목적이므로 트리의 깊이를 얕게(max_depth=5) 설정합니다.
        explainer_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=random_state)
        
        # 속도를 위해 Test 세트 중 2000개만 샘플링하여 학습 및 설명
        sample_size = min(2000, len(X_test))
        X_shap_train = X_test[:sample_size]
        cate_shap_train = cate_pred[:sample_size]
        
        explainer_model.fit(X_shap_train, cate_shap_train)

        # 2. SHAP 값 추출 및 시각화
        explainer = shap.TreeExplainer(explainer_model)
        shap_values = explainer.shap_values(X_shap_train)
        
        plt.figure(figsize=(8, 6))
        shap.summary_plot(shap_values, X_shap_train, feature_names=feature_names, show=False)
        plt.title("SHAP Summary Plot for Predicted CATE (Surrogate)", fontsize=14, fontweight='bold')
        plt.savefig(f"{output_dir}/4_shap_summary.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  -> SHAP 시각화 성공!")
    except Exception as e:
        print(f"  -> [!] SHAP 시각화 실패: {e}")

    # ---------------------------------------------------------
    # [Fig 5] CausalML Uplift Curves (Gain & Qini)
    # 실제 비즈니스 성과(타겟팅 효율) 검증
    # ---------------------------------------------------------
    try:
        from causalml.metrics import plot_gain
        print("[5/5] Uplift Evaluation Curves (Gain) 생성 중...")
        
        eval_df = pd.DataFrame({
            'y': Y_test.astype(float),
            't': T_test.astype(int),
            'CausalForest': cate_pred
        })
        
        plt.figure(figsize=(8, 6))
        plot_gain(eval_df, outcome_col='y', treatment_col='t')
        plt.title("Cumulative Gain Curve (Uplift)", fontsize=14, fontweight='bold')
        plt.savefig(f"{output_dir}/5_uplift_gain_curve.png", dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  -> [!] CausalML Uplift Curve 스킵: {e}")

    print(f"\n[*] 시각화 완료! 결과 이미지가 '{output_dir}/' 폴더에 저장되었습니다.")

plot_CausalForest(
    model=model,            # fast 모델 적용
    X_train=df_prep['X_train'],        # 샘플링된 train 적용
    T_train=df_prep['T_train'],
    X_test=df_prep['X_test'],    # Test는 전체 데이터 사용
    T_test=df_prep['T_test'],
    Y_test=df_prep['Y_test'],
    cate_pred=cate_pred,
    lb=lb,                  
    ub=ub,                  
    feature_names=df_prep['feature_names'],
    output_dir=SAVE_FOLDER_TEST
)


[*] GENERATING CAUSAL VISUALIZATIONS
[1/5] Overlap Assumption 체크 (Propensity Score)...
[2/5] 이질적 인과효과(CATE) 및 신뢰구간 분포 시각화...
[3/5] Feature Importance 추출 중...
[4/5] Surrogate Model을 활용한 SHAP Value 시각화...
  -> SHAP 시각화 성공!
[5/5] Uplift Evaluation Curves (Gain) 생성 중...

[*] 시각화 완료! 결과 이미지가 './Result/Test/' 폴더에 저장되었습니다.


<Figure size 800x600 with 0 Axes>